# COLAB_VERIFY — 제출 전 채점환경 모의 검증 노트북

이 노트북은 충남대 NLP 텀프로젝트(cnu-llm-bot)를 **마감(2026-06-12) 전에 채점 환경 그대로 한 번 끝까지 돌려보고**, 제출물에 문제가 없는지 눈으로 확인하기 위한 것입니다.

## 꼭 알아둘 점
- **채점(평가)은 `src/classifier.ipynb` 와 `chatbot.sh` 만 실행합니다.** 이 노트북은 그 두 가지가 잘 도는지 미리 확인하는 용도일 뿐, 제출물 자체가 아닙니다.
- 위에서 아래로 셀을 **순서대로 하나씩** 실행하세요. 중간에 빨간 에러(assert 실패)가 나면 거기서 멈추고 메시지를 읽으세요. 메시지에 어디서 왜 막혔는지 한글로 적혀 있습니다.
- **런타임 → 런타임 유형 변경 → T4 GPU** 로 먼저 바꿔두세요. (GPU 없으면 생성모델이 매우 느리거나 OOM 납니다.)
- 이 노트북은 코드를 받아오고(클론/드라이브), 자산(model·chroma_db)을 복원한 뒤, Task1/Task2/Task3 산출물 포맷을 검사하고, 마지막에 Gradio UI 공유링크까지 띄웁니다.

## 검사 흐름 요약
1. GPU 확인 → 2. 코드 받기 → 3. 자산 복원 확인(assert) → 4. 설치 → 5. Task1 분류 검증 → 6. 하이브리드/리랭커 점검 → 7. Task2 챗봇 검증 → 8. Task3 실시간 검증(옵션) → 9. UI 공유링크 → 10. 제출 전 체크리스트

## 1. GPU / 런타임 확인

아래 셀 출력에 `Tesla T4` 같은 GPU 이름이 보여야 합니다. `command not found` 나 빈 출력이 나오면 **런타임 → 런타임 유형 변경 → T4 GPU** 로 바꾼 뒤 다시 실행하세요.

In [ ]:
!nvidia-smi
import torch
print('torch =', torch.__version__, '| CUDA 사용가능 =', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\n[주의] GPU가 안 잡혔습니다. 런타임 유형을 T4 GPU 로 바꾸고 이 셀을 다시 실행하세요.')
    print('       CPU로도 분류기(Task1)는 돌지만, 생성모델(Task2/Task3)은 매우 느리거나 멈출 수 있습니다.')

## 2. 코드 가져오기 — (a) git clone  /  (b) 드라이브 마운트  중 택1

두 가지 방식을 모두 셀로 넣어두었습니다. **하나만 골라** 해당 줄의 주석(`#`)을 풀어 실행하세요.
- **(a) git clone**: 깔끔하게 깃허브에서 최신 코드를 받습니다. (자산 model/·chroma_db/ 는 들어있지 않으니 3번에서 복원)
- **(b) 드라이브 마운트**: 드라이브에 프로젝트 폴더(코드+자산)를 통째로 올려둔 경우 그걸 그대로 씁니다.

실행 후 `%cd` 로 **프로젝트 루트(cnu-llm-bot)** 안에 들어가 있어야 합니다. 마지막 줄의 `[OK] 프로젝트 루트 진입` 이 떠야 정상입니다.

In [ ]:
# ===== (a) git clone 방식 — 이 블록을 쓰려면 아래 두 줄의 주석을 푸세요 =====
# !git clone https://github.com/Longarden/cnu-llm-bot.git
# %cd cnu-llm-bot

# ===== (b) 드라이브 마운트 방식 — 이 블록을 쓰려면 아래 세 줄의 주석을 푸세요 =====
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/cnu-llm-bot   # ← 본인 드라이브의 실제 프로젝트 폴더 경로로 수정

# --- 공통: 프로젝트 루트에 제대로 들어왔는지 확인 ---
import os
from pathlib import Path
ROOT = Path.cwd()
must_have = ['chatbot.sh', 'requirements.txt', 'src', 'data']
missing = [m for m in must_have if not (ROOT / m).exists()]
assert not missing, (
    f'여기는 프로젝트 루트가 아닙니다(현재: {ROOT}). 없는 항목: {missing}. '
    '위에서 (a) 또는 (b) 중 하나의 주석을 풀어 실행했는지, %cd 경로가 맞는지 확인하세요.'
)
print('[OK] 프로젝트 루트 진입:', ROOT)

## 3. 자산 복원 확인 — model/ 와 chroma_db/ 실제 내용물까지 assert

채점은 `model/`(분류기 가중치)과 `chroma_db/`(벡터DB)가 있어야 돌아갑니다. 이 둘은 용량이 커서 깃에 안 들어있을 수 있습니다.

**없으면** 아래 둘 중 하나로 복원하세요:
- `!bash restore_assets.sh` (단, `restore_assets.sh` 안의 `MODEL_ID`/`CHROMA_ID` 를 본인 드라이브 파일ID로 먼저 채워야 함)
- 또는 드라이브에서 직접 복사 (예: `!cp -r /content/drive/MyDrive/cnu_model model` , `!cp -r /content/drive/MyDrive/cnu_chroma_db chroma_db`)

아래 셀은 **복원 후 실제 파일이 들어있는지**까지 검사합니다. 폴더만 있고 알맹이가 비면 빨간 에러로 멈춥니다.

In [ ]:
# 자산이 없으면 아래 한 줄의 주석을 풀어 복원한 뒤, 이 셀을 다시 실행하세요.
# !bash restore_assets.sh

from pathlib import Path
ROOT = Path.cwd()
MODEL = ROOT / 'model'
CHROMA = ROOT / 'chroma_db'

# (1) 폴더 존재
assert MODEL.exists(), 'model/ 폴더가 없습니다. restore_assets.sh 또는 드라이브에서 분류기를 복원하세요.'
assert CHROMA.exists(), 'chroma_db/ 폴더가 없습니다. restore_assets.sh 또는 드라이브에서 벡터DB를 복원하세요.'

# (2) model/ 알맹이: config.json + (safetensors 또는 model.bin) + 토크나이저
assert (MODEL / 'config.json').exists(), 'model/config.json 이 없습니다. 분류기 압축 해제가 덜 됐을 수 있습니다.'
has_weight = (MODEL / 'model.safetensors').exists() or (MODEL / 'model.bin').exists() or (MODEL / 'pytorch_model.bin').exists()
assert has_weight, 'model/ 안에 가중치(model.safetensors 또는 model.bin)가 없습니다. 분류기 복원이 불완전합니다.'
has_tok = (MODEL / 'tokenizer.json').exists() or (MODEL / 'tokenizer_config.json').exists() or (MODEL / 'vocab.txt').exists()
assert has_tok, 'model/ 안에 토크나이저 파일(tokenizer.json / tokenizer_config.json / vocab.txt)이 없습니다.'

# (3) chroma_db/ 알맹이: sqlite 파일
sqlite_files = list(CHROMA.glob('*.sqlite3')) + list(CHROMA.glob('*.sqlite'))
assert sqlite_files, 'chroma_db/ 안에 chroma.sqlite3 가 없습니다. 벡터DB 복원이 불완전합니다(폴더만 있고 DB가 비었습니다).'

print('[OK] model/ 내용:', sorted(p.name for p in MODEL.iterdir()))
print('[OK] chroma_db sqlite:', [p.name for p in sqlite_files])
print('[OK] 자산 복원 정상.')

## 4. 의존성 설치

`requirements.txt` 를 그대로 설치합니다. **torch 2.5.1 / torchvision 0.20.1 고정**이 핵심입니다.
- 콜랩 기본 torch/torchvision은 버전이 더 높아서, 안 맞추면 `transformers` 임포트가 `torchvision::nms` 에러로 통째로 깨집니다.
- 설치 후 **런타임이 재시작을 권하면(특히 torch 다운그레이드 시) 한 번 재시작**한 뒤, 2번(코드 진입)부터 다시 실행하세요.
- 설치는 몇 분 걸립니다. 빨간 경고(WARNING)는 무시해도 되지만, `ERROR` 로 실패하면 멈추고 메시지를 읽으세요.

In [ ]:
!pip install -q -r requirements.txt

# 핵심 버전 고정이 실제로 먹었는지 확인(틀리면 transformers 임포트가 깨질 수 있음)
import torch
print('torch =', torch.__version__, '(기대: 2.5.1)')
try:
    import torchvision
    print('torchvision =', torchvision.__version__, '(기대: 0.20.1)')
except Exception as e:
    print('torchvision 임포트 경고:', e)
import transformers
print('transformers =', transformers.__version__, '(기대: 4.46~4.48)')
if not torch.__version__.startswith('2.5.1'):
    print('\n[주의] torch가 2.5.1이 아닙니다. 런타임을 재시작(런타임 → 세션 다시 시작)하고 이 셀을 다시 실행하세요.')

## 5. Task1 검증 — 분류기 실행 + 출력 포맷 assert

`src/classifier.ipynb` 와 동일한 흐름(=`src/classifier.py` 의 `main()`)을 그대로 호출해 `data/test_cls.json` 을 예측하고 `outputs/cls_output.json` 을 만듭니다.

그 다음 산출물 포맷을 검사합니다:
- 결과가 **리스트**인지
- 각 항목에 **id · question · label** 키가 모두 있는지
- **label 이 0~4 정수**인지

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# classifier.ipynb 의 셀 흐름과 동일한 추론 로직(src/classifier.py 의 main)을 그대로 실행
import importlib
import src.classifier as classifier
importlib.reload(classifier)
classifier.main()

# --- 출력 포맷 검증 ---
CLS_OUT = ROOT / 'outputs' / 'cls_output.json'
assert CLS_OUT.exists(), f'{CLS_OUT} 가 생성되지 않았습니다. 위 분류기 실행 로그에서 에러를 확인하세요.'
with open(CLS_OUT, encoding='utf-8') as f:
    cls = json.load(f)

assert isinstance(cls, list) and len(cls) > 0, 'cls_output.json 이 비어있거나 리스트가 아닙니다.'
for i, row in enumerate(cls):
    for key in ('id', 'question', 'label'):
        assert key in row, f'{i}번째 항목에 "{key}" 키가 없습니다. (항목: {row})'
    assert isinstance(row['label'], int) and 0 <= row['label'] <= 4, (
        f'{i}번째 항목의 label 이 0~4 정수가 아닙니다: {row["label"]!r} (질문: {row["question"]})'
    )

# 라벨 분포가 한 곳에 쏠리지 않았는지 눈으로 확인(분류기가 한 라벨만 찍으면 의심)
from collections import Counter
dist = Counter(r['label'] for r in cls)
names = {0: '졸업요건', 1: '학교공지', 2: '학사일정', 3: '식단', 4: '통학/셔틀'}
print(f'[통과] Task1 포맷 정상 — 총 {len(cls)}건')
for lab in sorted(dist):
    print(f'   label {lab}({names[lab]}): {dist[lab]}건')
if len(dist) == 1:
    print('[주의] 모든 질문이 한 라벨로만 분류됐습니다. 분류기 가중치(model/)가 제대로 복원됐는지 의심하세요.')

## 6. 하이브리드 검색 / 리랭커 점검 (진단 셀)

검색 품질에 직접 영향을 주는 두 가지를 미리 확인합니다.

**(A) BM25(sparse) 초기화 여부** — 중요
이 프로젝트의 `retrieval/hybrid_retriever.py` 는 BM25 인덱스를 **자동으로 채우지 않습니다**(`init_bm25_from_db()` 를 명시 호출해야 채워짐). 안 채우면 sparse 검색이 빈 결과라 사실상 dense-only로 돕니다. 한 질문으로 sparse 가 실제 결과를 내는지 확인합니다.

**(B) 리랭커(CrossEncoder) 로드** — RERANK=1 환경에서 `BAAI/bge-reranker-v2-m3` 가 OOM 없이 올라오는지 try/except 로 확인합니다. 실패하면 RERANK=0 으로 돌리라고 안내합니다.

이 셀은 진단용이며 실패해도 채점이 곧장 깨지는 건 아니지만, **답변 품질을 좌우**하므로 결과를 꼭 눈으로 보세요.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

Q = '컴퓨터인공지능학부 졸업하려면 뭐가 필요해?'

# (A) BM25 sparse 점검 -------------------------------------------------
import retrieval.hybrid_retriever as hr
print('[A] init_bm25_from_db() 호출 전 sparse 결과 수:', len(hr._sparse_search(Q, 5)))
try:
    hr.init_bm25_from_db()  # chroma_db 전체 문서로 BM25 채움
except Exception as e:
    print('[A][경고] BM25 초기화 실패:', e)
sparse_after = hr._sparse_search(Q, 5)
print('[A] init 후 sparse 결과 수:', len(sparse_after))
if not sparse_after:
    print('[A][주의] sparse 가 비었습니다 → 하이브리드가 dense-only로 동작합니다.')
    print('         chatbot.sh(=gen_chat_output.py) 경로는 BM25를 자동 초기화하지 않으므로,')
    print('         검색이 dense에만 의존합니다. 답변이 약하면 이 부분을 의심하세요.')
else:
    print('[A][통과] BM25 sparse 검색 동작. 상위 1건 미리보기:')
    print('   ', (sparse_after[0].get('original_text', '') or '')[:120])

# (B) 리랭커 로드 점검 (RERANK=1) -------------------------------------
import os
os.environ['RERANK'] = '1'
try:
    import retrieval.reranker as rr
    ce = rr._get_cross_encoder()  # CrossEncoder 1회 로드
    if not ce:
        print('[B][주의] 리랭커 로드 실패(폴백 상태). RERANK=0 으로 두고 진행하세요. (위 [reranker] 로그 확인)')
    else:
        ranked = rr.rerank(Q, [d for d in sparse_after] or [{'text': '테스트'}], top_k=2)
        print('[B][통과] 리랭커(bge-reranker-v2-m3) 로드 + 동작 OK. 재정렬 결과 수:', len(ranked))
except Exception as e:
    print('[B][주의] 리랭커 점검 중 오류:', e)
    print('         OOM 등으로 실패하면 Task2 실행 전에  os.environ["RERANK"]="0"  으로 끄고 재시도하세요.')

## 7. Task2 검증 — chatbot.sh 와 동일하게 챗봇 산출물 생성 + 포맷/거절 점검

채점은 `chatbot.sh` 를 그대로 실행합니다. 그 스크립트의 1단계가 `python src/gen_chat_output.py` 로 `outputs/chat_output.json` 을 만드는 부분입니다.

여기서는 **채점과 동일하게** `RERANK=1` 환경에서 챗봇 배치 추론을 돌립니다.
- 아래 셀은 `chatbot.sh` 의 핵심인 `gen_chat_output.py` 를 직접 실행합니다(전체 `chatbot.sh` 는 마지막에 UI까지 띄우므로, 산출물 검사 단계에서는 이 부분만 분리해 돌립니다).
- **전체 `chatbot.sh` 를 채점처럼 통째로 돌려보고 싶으면** 9번(UI) 셀에서 `!bash chatbot.sh` 를 쓰세요. 그게 1)chat_output 2)realtime_output 3)UI 를 한 번에 합니다.

생성 후: `id · user · model` 키 확인 + `model` 이 빈문자열/None 이 아닌지 + 5개 유형 샘플 응답을 표로 보고 **거절 도배가 아닌지** 눈으로 확인합니다.

**시간 주의:** 생성모델(Qwen2.5-7B 4bit) 첫 로드 + 16문항 생성은 몇 분 걸립니다. T4 기준 정상입니다.

In [ ]:
import os, sys, json, importlib
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 채점과 동일한 환경: 로컬 생성 + 실시간 라이브크롤 ON + 리랭커 ON
os.environ['GEN_BACKEND'] = 'local'
os.environ['CHAT_REALTIME'] = '1'
os.environ.setdefault('RERANK', '1')
# 리랭커가 6번에서 실패했다면 아래 줄의 주석을 풀어 끄고 돌리세요:
# os.environ['RERANK'] = '0'

import src.gen_chat_output as gco
importlib.reload(gco)
gco.main()  # = chatbot.sh 의 1단계(python src/gen_chat_output.py)

# --- 출력 포맷 검증 ---
CHAT_OUT = ROOT / 'outputs' / 'chat_output.json'
assert CHAT_OUT.exists(), f'{CHAT_OUT} 가 생성되지 않았습니다. 위 로그에서 에러를 확인하세요.'
with open(CHAT_OUT, encoding='utf-8') as f:
    chat = json.load(f)
assert isinstance(chat, list) and len(chat) > 0, 'chat_output.json 이 비어있거나 리스트가 아닙니다.'
for i, row in enumerate(chat):
    for key in ('id', 'user', 'model'):
        assert key in row, f'{i}번째 항목에 "{key}" 키가 없습니다. (항목: {row})'
    assert row['model'] is not None and str(row['model']).strip() != '', (
        f'{i}번째 항목의 model 답변이 비었습니다(질문: {row["user"]}). 생성이 실패했을 수 있습니다.'
    )
    assert not str(row['model']).startswith('생성 오류'), (
        f'{i}번째 답변이 "생성 오류" 로 시작합니다(질문: {row["user"]}). 모델 로드/생성 실패입니다.'
    )
print(f'[통과] Task2 포맷 정상 — 총 {len(chat)}건, 빈 답변/생성오류 없음')

In [ ]:
# 5개 유형(졸업/공지/학사일정/식단/셔틀) 샘플 응답을 표로 보고 '거절 도배'가 아닌지 눈으로 확인
import json
from pathlib import Path
ROOT = Path.cwd()
with open(ROOT / 'outputs' / 'chat_output.json', encoding='utf-8') as f:
    chat = json.load(f)

# 각 유형을 대표하는 키워드로 해당 질문 하나씩 골라 본다
type_keywords = {
    '졸업요건': ['졸업', '학점', '전공', '복수전공', '전과'],
    '학교공지': ['공지', '장학', '비교과'],
    '학사일정': ['수강신청', '정정', '기말', '휴학', '계절학기', '일정'],
    '식단':    ['학식', '식단', '메뉴', '점심', '학생회관'],
    '통학/셔틀': ['셔틀', '버스', '정류장', '통학', '노선'],
}
REJECT_HINTS = ['찾지 못', '자료가 없', '정보를 가져오지 못', '죄송', '확인되지 않', '제공할 수 없']

def pick(keys):
    for row in chat:
        if any(k in row['user'] for k in keys):
            return row
    return None

rows_md = ['| 유형 | 질문 | 답변(앞 120자) | 거절추정 |', '|---|---|---|---|']
reject_count = 0
for tname, keys in type_keywords.items():
    r = pick(keys)
    if not r:
        rows_md.append(f'| {tname} | (해당 질문 없음) | - | - |')
        continue
    ans = str(r['model']).replace('\n', ' ').strip()
    looks_reject = any(h in ans for h in REJECT_HINTS)
    reject_count += int(looks_reject)
    q = r['user'][:24]
    rows_md.append(f'| {tname} | {q} | {ans[:120]} | {"예" if looks_reject else "아니오"} |')

try:
    from IPython.display import Markdown, display
    display(Markdown('\n'.join(rows_md)))
except Exception:
    print('\n'.join(rows_md))

print(f'\n5개 유형 중 거절로 보이는 응답: {reject_count}개')
if reject_count >= 3:
    print('[주의] 절반 이상이 거절로 보입니다. 식단/셔틀은 라이브크롤(CHAT_REALTIME=1), 그 외는 chroma_db 복원 여부를 의심하세요.')
else:
    print('[통과] 거절 도배는 아닙니다. 다만 각 답변 내용이 질문과 맞는지 직접 한 번 읽어보세요.')

## 8. Task3 검증 (옵션, +30점) — realtime_output 포맷 + 라이브 데이터 점검

`chatbot.sh` 의 2단계(`python src/realtime_model.py`)가 만드는 `outputs/realtime_output.json` 을 검사합니다.
- 포맷: `id · user · model`, `model` 이 비어있지 않음
- **오늘자 라이브 데이터가 실제로 들어왔는지** — 폴백 문구("실시간 정보를 가져오지 못했습니다")가 도배돼 있지 않은지 확인

네트워크/크롤 사정에 따라 일부 폴백은 정상입니다. 다만 **전부 폴백이면** Task3 점수가 안 나오니 재시도하세요.

In [ ]:
import os, sys, json, importlib
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['GEN_BACKEND'] = 'local'

import src.realtime_model as rtm
importlib.reload(rtm)
rtm.main()  # = chatbot.sh 의 2단계(python src/realtime_model.py)

RT_OUT = ROOT / 'outputs' / 'realtime_output.json'
assert RT_OUT.exists(), f'{RT_OUT} 가 생성되지 않았습니다. 위 로그에서 크롤/생성 에러를 확인하세요.'
with open(RT_OUT, encoding='utf-8') as f:
    rt = json.load(f)
assert isinstance(rt, list) and len(rt) > 0, 'realtime_output.json 이 비어있거나 리스트가 아닙니다.'
for i, row in enumerate(rt):
    for key in ('id', 'user', 'model'):
        assert key in row, f'{i}번째 항목에 "{key}" 키가 없습니다. (항목: {row})'
    assert row['model'] is not None and str(row['model']).strip() != '', (
        f'{i}번째 항목의 model 답변이 비었습니다(질문: {row["user"]}).'
    )

FALLBACK = '실시간 정보를 가져오지 못했습니다'
fb = sum(1 for r in rt if FALLBACK in str(r['model']))
print(f'[통과] Task3 포맷 정상 — 총 {len(rt)}건, 폴백 {fb}건')
for r in rt:
    print('  -', r['user'][:30], '→', str(r['model']).replace(chr(10), ' ')[:90])
if fb == len(rt):
    print('\n[주의] 모든 답변이 폴백입니다(라이브 크롤 0건). 네트워크 상태 확인 후 이 셀을 다시 실행하세요.')
else:
    print('\n[통과] 일부 이상 라이브 데이터 반영됨. 식단/셔틀/공지 답에 오늘자 정보가 보이는지 직접 확인하세요.')

## 9. UI 실행 — Gradio 공유링크 + 시연 영상 촬영 팁

채점은 `chatbot.sh` 의 3단계로 Gradio UI를 띄웁니다. 콜랩에서는 출력 로그에 뜨는 **`*.gradio.live`** 공유링크로 접속합니다.

두 가지 방법 중 택1:
- **(권장, 채점과 동일)** `!bash chatbot.sh` — 1)chat_output 2)realtime_output 3)UI 를 한 번에. (이미 7·8번을 돌렸다면 산출물은 덮어쓰기됩니다)
- **(UI만 빠르게)** `src/chatbot_ui.py` 의 `launch_app(share=True)` 만 호출.

**중요:** 콜랩 셀이 UI 프로세스에 잡혀 계속 실행 상태가 됩니다. 링크가 뜨면 그걸로 접속해 시연하고, 끝나면 셀을 정지(■)하세요.

**2분 시연 영상 촬영 팁** — 5개 유형을 골고루 질문해 분류 뱃지+응답을 보여주세요:
1. 졸업: "컴퓨터인공지능학부 졸업하려면 뭐가 필요해?"
2. 공지: "최근 학교 공지 뭐 올라왔어?"
3. 학사일정: "수강신청 정정 기간이 언제예요?"
4. 식단: "오늘 학식 메뉴가 뭐예요?"
5. 셔틀: "궁동행 셔틀 배차 간격이 어떻게 돼?"

각 답변 상단의 유형 뱃지(예: 🎓 졸업요건)가 질문과 맞게 뜨는지도 영상에 담으세요.

In [ ]:
import os
os.environ['GRADIO_SHARE'] = '1'   # 콜랩 공유링크(*.gradio.live) 발급
os.environ['GEN_BACKEND'] = 'local'
os.environ['CHAT_REALTIME'] = '1'

# ===== (권장) 채점과 동일하게 chatbot.sh 통째 실행 — 아래 한 줄 주석 해제 =====
# !bash chatbot.sh

# ===== (UI만 빠르게) 위를 안 쓰고 UI만 띄우려면 아래 두 줄 주석 해제 =====
# import sys; sys.path.insert(0, os.getcwd())
# from src.chatbot_ui import launch_app; launch_app(share=True)

print('위 둘 중 하나의 주석을 풀어 실행하세요. 로그에 https://....gradio.live 링크가 뜨면 그걸로 접속해 시연합니다.')

## 10. 제출 전 최종 체크리스트

아래 항목을 하나씩 직접 확인하고 제출하세요.

- [ ] **제출 파일명**: `Termproject_장정원` (폴더/zip 이름). `scripts/package_submission.sh` 의 NAME 을 본인 이름으로.
- [ ] **디렉터리 규칙 준수**: PDF에 지정된 빨간 경로(과제 명세의 디렉토리 구조) 그대로인지. 특히 `src/classifier.ipynb`, `chatbot.sh`, `outputs/`, `data/`, `model/`, `chroma_db/` 위치 확인.
- [ ] **requirements.txt 포함** — 조교가 `pip install -r requirements.txt` 로 환경을 재현합니다. (torch 2.5.1 / torchvision 0.20.1 / transformers<4.49 핀 유지)
- [ ] **자산 복원 안내**: model/·chroma_db/ 가 zip 에 없다면 `restore_assets.sh` 의 드라이브 ID가 채워져 있고 링크 공유가 "링크 있는 모든 사용자: 뷰어" 인지.
- [ ] **채점 재현 확인**: 새 콜랩에서 이 노트북을 처음부터 끝까지 한 번 무중단 실행했고, 5·7·8번 assert가 모두 통과했는가.
- [ ] **outputs 산출물**: `cls_output.json`(Task1) / `chat_output.json`(Task2) / `realtime_output.json`(Task3) 포맷·내용 확인. (제출 zip의 outputs/는 비어 있어도 됨 — 채점 시 자동 생성)
- [ ] **시연 영상(2분)**: 5개 유형 질문 + 분류 뱃지 + 응답이 담겼는가. `*.gradio.live` UI로 촬영.
- [ ] **발표자료(5분)**: 파이프라인(분류→라우팅 RAG/라이브크롤→생성), 하이브리드+리랭커, 거절/CRAG, Task3 라이브성 포인트 포함.

여기까지 모두 체크되면 제출 준비 완료입니다. 마감 2026-06-12 전에 여유를 두고 한 번 더 새 런타임에서 처음부터 돌려보세요.